<a href="https://colab.research.google.com/github/Adityagujjar-08/dataviz-exercises-adityapatil/blob/main/lecture07_exercise_Aditya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv(r'/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
import plotly.express as px

# Filter to fossil sources only
df_fossil = df[df['Source_Type'] == 'Fossil'].copy()

# Create a treemap: Region -> Country -> Source
# Color encodes the specific fossil source (Coal / Oil / Natural Gas)
fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': '#E69F00',        # CVD-safe orange
        'Oil': '#0072B2',         # CVD-safe blue
        'Natural Gas': '#009E73', # CVD-safe green
        '(?' : '#D3D3D3'          # Ensure non-matching parent branches fall back to grey
    },
    title="North America and Asia Dominate Fossil Fuel Consumption (TWh)"
)

# Requirements: Use a neutral grey for the parent nodes and display labels and values
fig1.update_traces(
    marker=dict(colorscale=[[0, '#D3D3D3'], [1, '#D3D3D3']], cmid=0),
    textinfo="label+value"
)

fig1.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
import plotly.express as px

# Filter to low-carbon and aggregate TWh by country
df_low_carbon = df[df['Source_Type'] == 'Low-carbon'].copy()
df_low_carbon_agg = df_low_carbon.groupby('Country', as_index=False)['TWh'].sum()

# Add a dummy root node for the Treemap
df_low_carbon_agg['All'] = 'Low-carbon'

# Treemap
fig2_tree = px.treemap(
    df_low_carbon_agg,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#56B4E9'], # CVD-safe sky blue
    title="Low-Carbon Energy Distribution by Country"
)
fig2_tree.update_traces(textinfo="label+value")
fig2_tree.show()

# Bar chart: sorted by TWh, horizontal orientation
df_bar_sorted = df_low_carbon_agg.sort_values(by='TWh', ascending=True)

fig2_bar = px.bar(
    df_bar_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'], # CVD-safe blue
    title="United States Leads in Total Low-Carbon Energy Generation (TWh)"
)
fig2_bar.update_layout(yaxis={'categoryorder': 'total ascending'})
fig2_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

**The horizontal bar chart tells the ranking story much more clearly.** On the bar chart, all bars share a common baseline aligned along the y-axis, making immediate side-by-side length comparisons effortless and precise. In contrast, the treemap encodes the values using 2D area blocks of varying aspect ratios, which are perceptually harder to rank and compare accurately at a glance.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
import plotly.express as px

# 1. Reuse df_fossil from Task 1
df_fossil = df[df['Source_Type'] == 'Fossil'].copy()

# 2. Aggregate TWh by Region x Source
df_region_fossil = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()

# 3. Calculate each region's total fossil TWh to convert to share (%)
region_totals = df_region_fossil.groupby('Region')['TWh'].transform('sum')
df_region_fossil['Share_pct'] = (df_region_fossil['TWh'] / region_totals) * 100

# 4. Determine sorting: Order regions by Coal share (highest at top)
coal_shares = df_region_fossil[df_region_fossil['Source'] == 'Coal'].set_index('Region')['Share_pct']
sorted_regions = coal_shares.sort_values(ascending=True).index.tolist()

# 5. Build horizontal 100% stacked bar chart
fig3 = px.bar(
    df_region_fossil,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    category_orders={'Region': sorted_regions, 'Source': ['Coal', 'Oil', 'Natural Gas']},
    color_discrete_map={
        'Coal': '#E69F00',         # Same CVD-safe orange as in Task 1
        'Oil': '#999999',          # Grey 1
        'Natural Gas': '#CCCCCC'   # Grey 2
    },
    title="Asia is the Most Coal-Dependent Region, Representing ~56% of Its Fossil Mix",
    labels={'Share_pct': 'Fossil Share (%)'}
)

# 6. Add percentage labels on the Coal segments only
fig3.update_traces(
    texttemplate='%{x:.1f}%',
    textposition='inside',
    selector=dict(name='Coal')
)
# Remove labels from greyed segments
fig3.update_traces(texttemplate='', selector=dict(name='Oil'))
fig3.update_traces(texttemplate='', selector=dict(name='Natural Gas'))

fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Treemap:** Helps identify which countries consume the most fossil energy in absolute terms (TWh). It clearly shows that countries like China and the US have much higher consumption than others.

- **Stacked Bar Chart:** Helps compare how dependent each region is on coal, gas and oil. Since the chart shows percentages, it's easier to compare the energy mix between regions, even if their total consumption is very different.